# Walk-Forward Analytics
Shows how model performance evolves as the training window grows over time.

**Data source:** `reports/walkforward/combined_detail.csv`  
**Sections:**
1. Setup and data load
2. Single ticker analysis — tables and charts
3. All tickers combined — tables and charts


## 1. Setup

In [1]:
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from IPython.display import display

# ── Load data ──────────────────────────────────────────────────────────────
DATA_PATH = "reports/walkforward/combined_detail.csv"

df = pd.read_csv(DATA_PATH)

# train_end is our human-readable X axis label for each window
df["train_end"] = pd.to_datetime(df["train_end"])
df["label"]     = df["train_end"].dt.strftime("%b '%y")

MODELS  = sorted(df["model"].unique())
TICKERS = sorted(df["ticker"].unique())

MODEL_COLORS = {
    "gradient_boosting":   "#BA7517",
    "logistic_regression": "#1D9E75",
    "random_forest":       "#534AB7",
}

MODEL_LABELS = {
    "gradient_boosting":   "Gradient Boosting",
    "logistic_regression": "Logistic Regression",
    "random_forest":       "Random Forest",
}

print(f"Loaded {len(df):,} rows")
print(f"Tickers : {TICKERS}")
print(f"Models  : {MODELS}")
print(f"Windows : {sorted(df['window'].unique())}")
print(f"Training cutoffs: {sorted(df['label'].unique())}")

Loaded 210 rows
Tickers : ['AAPL', 'AMC', 'AMZN', 'GME', 'GOOGL', 'MSFT', 'NFLX', 'SPY', 'SYY', 'USFD']
Models  : ['gradient_boosting', 'logistic_regression', 'random_forest']
Windows : [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7)]
Training cutoffs: ["Aug '25", "Dec '23", "Feb '23", "Jul '23", "Mar '25", "May '24", "Oct '24"]


---
## 2. Single Ticker Analysis
Set `TICKER` below to the symbol you want to explore.


In [2]:
TICKER = TICKERS[0]   # change to any ticker in your dataset e.g. "MSFT"

t = df[df["ticker"] == TICKER].copy()
print(f"Analysing {TICKER}  —  {len(t):,} rows across {t['window'].nunique()} windows")

Analysing AAPL  —  21 rows across 7 windows


### 2a. Scores by Window and Model

In [3]:
table = (
    t[["window","label","model","train_rows","test_rows","accuracy","roc_auc","precision","recall","f1"]]
    .copy()
    .rename(columns={
        "window":     "Window",
        "label":      "Train cutoff",
        "model":      "Model",
        "train_rows": "Train rows",
        "test_rows":  "Test rows",
        "accuracy":   "Accuracy",
        "roc_auc":    "ROC AUC",
        "precision":  "Precision",
        "recall":     "Recall",
        "f1":         "F1",
    })
)
table["Model"]    = table["Model"].map(MODEL_LABELS)
table["Accuracy"] = table["Accuracy"].map("{:.1%}".format)
table["ROC AUC"]  = table["ROC AUC"].map("{:.3f}".format)
table["Precision"]= table["Precision"].map("{:.3f}".format)
table["Recall"]   = table["Recall"].map("{:.3f}".format)
table["F1"]       = table["F1"].map("{:.3f}".format)

display(table.sort_values(["Window","Model"]).reset_index(drop=True))

,Window,Train cutoff,Model,Train rows,Test rows,Accuracy,ROC AUC,Precision,Recall,F1
0,1,Feb '23,Gradient Boosting,1245,207,56.5%,0.495,0.572,0.861,0.688
1,1,Feb '23,Logistic Regression,1245,207,55.1%,0.481,0.556,0.948,0.701
2,1,Feb '23,Random Forest,1245,207,57.0%,0.452,0.567,0.957,0.712
3,2,Jul '23,Gradient Boosting,1349,207,50.7%,0.511,0.513,0.726,0.602
4,2,Jul '23,Logistic Regression,1349,207,51.7%,0.418,0.515,0.972,0.673
5,2,Jul '23,Random Forest,1349,207,52.7%,0.489,0.521,0.924,0.667
6,3,Dec '23,Gradient Boosting,1453,207,49.8%,0.484,0.526,0.796,0.634
7,3,Dec '23,Logistic Regression,1453,207,52.2%,0.356,0.537,0.894,0.671
8,3,Dec '23,Random Forest,1453,207,51.7%,0.460,0.534,0.903,0.671
9,4,May '24,Gradient Boosting,1557,207,56.5%,0.527,0.596,0.756,0.667


### 2b. Best Model per Window

In [4]:
best_rows = []
prev_auc  = None

for win in sorted(t["window"].unique()):
    win_df   = t[t["window"] == win]
    best_idx = win_df["roc_auc"].idxmax()
    best     = win_df.loc[best_idx]

    improved = None
    if prev_auc is not None:
        improved = "▲ Yes" if best["roc_auc"] >= prev_auc else "▼ No"
    else:
        improved = "— First"

    best_rows.append({
        "Window":      int(best["window"]),
        "Train cutoff":best["label"],
        "Train rows":  int(best["train_rows"]),
        "Test rows":   int(best["test_rows"]),
        "Best model":  MODEL_LABELS[best["model"]],
        "ROC AUC":     f"{best['roc_auc']:.3f}",
        "Accuracy":    f"{best['accuracy']:.1%}",
        "Improved?":   improved,
    })
    prev_auc = best["roc_auc"]

display(pd.DataFrame(best_rows))

,Window,Train cutoff,Train rows,Test rows,Best model,ROC AUC,Accuracy,Improved?
0,1,Feb '23,1245,207,Gradient Boosting,0.495,56.5%,— First
1,2,Jul '23,1349,207,Gradient Boosting,0.511,50.7%,▲ Yes
2,3,Dec '23,1453,207,Gradient Boosting,0.484,49.8%,▼ No
3,4,May '24,1557,207,Gradient Boosting,0.527,56.5%,▲ Yes
4,5,Oct '24,1660,207,Gradient Boosting,0.488,54.6%,▼ No
5,6,Mar '25,1764,207,Gradient Boosting,0.547,54.6%,▲ Yes
6,7,Aug '25,1868,207,Gradient Boosting,0.590,58.9%,▲ Yes


### 2c. ROC AUC and Accuracy Across Windows

In [5]:
metric = "roc_auc"    # change to "accuracy" to switch metric

fig = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.08,
    subplot_titles=["ROC AUC by Model", "Accuracy by Model"]
)

labels = t.drop_duplicates("window").sort_values("window")["label"].tolist()

for model in MODELS:
    sub   = t[t["model"] == model].sort_values("window")
    color = MODEL_COLORS[model]
    label = MODEL_LABELS[model]

    fig.add_trace(go.Scatter(
        x=sub["label"], y=sub["roc_auc"],
        mode="lines+markers", name=label,
        line=dict(color=color, width=2), marker=dict(size=8),
        legendgroup=label,
    ), row=1, col=1)

    fig.add_trace(go.Scatter(
        x=sub["label"], y=sub["accuracy"].mul(100),
        mode="lines+markers", name=label,
        line=dict(color=color, width=2), marker=dict(size=8),
        legendgroup=label, showlegend=False,
    ), row=2, col=1)

# Baselines
fig.add_hline(y=0.50,  line_dash="dash", line_color="#888780", row=1, col=1,
              annotation_text="0.50 baseline", annotation_position="top left")
fig.add_hline(y=50,    line_dash="dash", line_color="#888780", row=2, col=1,
              annotation_text="50% baseline",  annotation_position="top left")

fig.update_layout(
    title=f"{TICKER} — Model Performance Across Walk-Forward Windows",
    height=560, hovermode="x unified",
    legend=dict(orientation="h", y=1.06)
)
fig.update_yaxes(title_text="ROC AUC",     row=1, col=1, range=[0.35, 0.65])
fig.update_yaxes(title_text="Accuracy (%)",row=2, col=1, range=[35, 65])
fig.show()

### 2d. Prediction Bias Across Windows

In [6]:
# Prediction bias shows whether a model skews toward predicting UP or DOWN
# as the training window grows. A healthy model should call both directions
# in roughly the same proportion as the actuals.

bias_rows = []
for _, row in t.iterrows():
    total = row["pred_up_count"] + row["pred_down_count"]
    if total == 0:
        continue
    bias_rows.append({
        "label":  row["label"],
        "window": row["window"],
        "Model":  MODEL_LABELS[row["model"]],
        "UP":     round(row["pred_up_count"]  / total * 100, 1),
        "DOWN":   round(row["pred_down_count"] / total * 100, 1),
        "Actual UP": round(row["actual_up_count"] / total * 100, 1),
    })

bias_df = pd.DataFrame(bias_rows)

fig = go.Figure()

for model_name in [MODEL_LABELS[m] for m in MODELS]:
    sub = bias_df[bias_df["Model"] == model_name]
    color = [v for k,v in MODEL_COLORS.items() if MODEL_LABELS[k] == model_name][0]
    fig.add_trace(go.Scatter(
        x=sub["label"], y=sub["UP"],
        mode="lines+markers", name=f"{model_name}",
        line=dict(color=color, width=2), marker=dict(size=7),
    ))

# Actual UP rate for reference
actual_up = bias_df.drop_duplicates("window").sort_values("window")
fig.add_trace(go.Scatter(
    x=actual_up["label"], y=actual_up["Actual UP"],
    mode="lines", name="Actual UP rate",
    line=dict(color="#888780", width=1.5, dash="dot"),
))

fig.update_layout(
    title=f"{TICKER} — % of Predictions Calling UP by Window",
    xaxis_title="Training cutoff", yaxis_title="% Predicted UP",
    height=400, hovermode="x unified",
    legend=dict(orientation="h", y=1.1),
    yaxis=dict(range=[30, 80])
)
fig.add_hline(y=50, line_dash="dash", line_color="#D85A30", opacity=0.4)
fig.show()

---
## 3. All Tickers Combined


### 3a. Best Model per Window — All Tickers

In [7]:
all_best = []

for ticker in TICKERS:
    prev_auc = None
    t_df = df[df["ticker"] == ticker]

    for win in sorted(t_df["window"].unique()):
        win_df   = t_df[t_df["window"] == win]
        best_idx = win_df["roc_auc"].idxmax()
        best     = win_df.loc[best_idx]

        improved = None
        if prev_auc is not None:
            improved = "▲ Yes" if best["roc_auc"] >= prev_auc else "▼ No"
        else:
            improved = "— First"

        all_best.append({
            "Ticker":      ticker,
            "Window":      int(best["window"]),
            "Train cutoff":best["label"],
            "Best model":  MODEL_LABELS[best["model"]],
            "ROC AUC":     round(best["roc_auc"], 3),
            "Accuracy":    f"{best['accuracy']:.1%}",
            "Improved?":   improved,
        })
        prev_auc = best["roc_auc"]

all_best_df = pd.DataFrame(all_best)
display(all_best_df)

,Ticker,Window,Train cutoff,Best model,ROC AUC,Accuracy,Improved?
0,AAPL,1,Feb '23,Gradient Boosting,0.495,56.5%,— First
1,AAPL,2,Jul '23,Gradient Boosting,0.511,50.7%,▲ Yes
2,AAPL,3,Dec '23,Gradient Boosting,0.484,49.8%,▼ No
3,AAPL,4,May '24,Gradient Boosting,0.527,56.5%,▲ Yes
4,AAPL,5,Oct '24,Gradient Boosting,0.488,54.6%,▼ No
...,...,...,...,...,...,...,...
65,USFD,3,Dec '23,Gradient Boosting,0.519,50.7%,▼ No
66,USFD,4,May '24,Random Forest,0.552,49.3%,▲ Yes
67,USFD,5,Oct '24,Gradient Boosting,0.596,53.1%,▲ Yes
68,USFD,6,Mar '25,Gradient Boosting,0.527,50.7%,▼ No


### 3b. Average AUC per Model — All Tickers

In [8]:
avg_auc = (
    df.groupby(["ticker", "model"])
    .agg(
        Avg_AUC      = ("roc_auc",  "mean"),
        Avg_Accuracy = ("accuracy", "mean"),
        Windows      = ("window",   "nunique"),
    )
    .round(3)
    .reset_index()
    .rename(columns={"ticker": "Ticker", "model": "Model"})
)
avg_auc["Model"]        = avg_auc["Model"].map(MODEL_LABELS)
avg_auc["Avg_Accuracy"] = avg_auc["Avg_Accuracy"].map("{:.1%}".format)

display(avg_auc.sort_values(["Ticker","Avg_AUC"], ascending=[True, False]).reset_index(drop=True))

,Ticker,Model,Avg_AUC,Avg_Accuracy,Windows
0,AAPL,Gradient Boosting,0.520,54.5%,7
1,AAPL,Random Forest,0.480,54.5%,7
2,AAPL,Logistic Regression,0.427,52.3%,7
3,AMC,Gradient Boosting,0.510,55.8%,7
4,AMC,Logistic Regression,0.493,57.9%,7
5,AMC,Random Forest,0.478,56.0%,7
6,AMZN,Logistic Regression,0.522,53.5%,7
7,AMZN,Gradient Boosting,0.505,50.4%,7
8,AMZN,Random Forest,0.497,51.6%,7
9,GME,Gradient Boosting,0.522,52.7%,7


### 3c. ROC AUC Trend Across Windows — All Tickers

In [9]:
fig = px.line(
    df.assign(Model=df["model"].map(MODEL_LABELS)),
    x="label", y="roc_auc",
    color="Model", facet_row="ticker",
    color_discrete_map={MODEL_LABELS[k]: v for k,v in MODEL_COLORS.items()},
    markers=True,
    title="ROC AUC Across Walk-Forward Windows — All Tickers",
    labels={"label": "Training cutoff", "roc_auc": "ROC AUC", "ticker": "Ticker"},
    height=320 * len(TICKERS),
)
fig.add_hline(y=0.50, line_dash="dash", line_color="#888780",
              annotation_text="0.50 baseline")
fig.update_layout(hovermode="x unified", legend=dict(orientation="h", y=1.02))
fig.update_xaxes(tickangle=45)
fig.update_yaxes(range=[0.35, 0.65])
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.show()

### 3d. Which Model Wins Most Often — All Tickers

In [10]:
# For each ticker and window, find which model had the highest AUC.
# Then count how many windows each model won per ticker.

winner_rows = []
for ticker in TICKERS:
    t_df = df[df["ticker"] == ticker]
    for win in sorted(t_df["window"].unique()):
        win_df = t_df[t_df["window"] == win]
        best   = win_df.loc[win_df["roc_auc"].idxmax(), "model"]
        winner_rows.append({"Ticker": ticker, "Model": MODEL_LABELS[best]})

winners = (
    pd.DataFrame(winner_rows)
    .groupby(["Ticker", "Model"])
    .size()
    .reset_index(name="Windows Won")
)

fig = px.bar(
    winners, x="Ticker", y="Windows Won", color="Model",
    barmode="group",
    color_discrete_map={MODEL_LABELS[k]: v for k,v in MODEL_COLORS.items()},
    title="Number of Windows Won by Model — All Tickers",
    height=420,
)
fig.update_layout(legend=dict(orientation="h", y=1.1))
fig.show()